# Exploration et préparation des données de délinquance

Ce notebook constitue la première étape du projet **Crime Analytics Dashboard**.

Son objectif est de :

- comprendre la structure du jeu de données ;
- vérifier sa qualité ;
- préparer les variables nécessaires ;
- sélectionner le périmètre de la France métropolitaine ;
- exporter un fichier propre destiné à la base SQL et au dashboard Power BI.

Les analyses métier, les KPI et les visualisations finales seront réalisés ensuite avec **SQL** et **Power BI**.

## 1. Présentation du jeu de données

Le site [data.gouv.fr](https://www.data.gouv.fr/) met à disposition les données publiques produites par les administrations françaises.

Le jeu de données utilisé provient des bases statistiques départementales de la délinquance enregistrée par la police et la gendarmerie nationales. Il rassemble, depuis 2016 :

- le nombre de faits constatés pour différentes catégories d'infractions ;
- les codes des départements et des régions ;
- la population et le nombre de logements ;
- les taux pour mille associés aux indicateurs.

Les données sont disponibles sur la page suivante :  
[Base statistique de la délinquance enregistrée](https://www.data.gouv.fr/datasets/bases-statistiques-communale-departementale-et-regionale-de-la-delinquance-enregistree-par-la-police-et-la-gendarmerie-nationales/)

## 2. Importation des données

In [3]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

# Le notebook est prévu pour être placé dans le dossier notebooks/
DATA_PATH = Path("../data/raw/donnee-dep-data.gouv-2024-geographie2025-produit-le2025-06-04.csv")
OUTPUT_PATH = Path("../data/processed/delinquance_metropole.csv")

In [4]:
delits = pd.read_csv(
    DATA_PATH,
    sep=";",
    dtype={"Code_departement": "string"}
)

delits.head()

,Code_departement,Code_region,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime
0,01,84,2016,Homicides,Victime,5,"0,0078318",638425,2016,308491,2016
1,02,32,2016,Homicides,Victime,10,"0,0186520",536136,2016,264180,2016
2,03,84,2016,Homicides,Victime,4,"0,0117861",339384,2016,206980,2016
3,04,93,2016,Homicides,Victime,2,"0,0123028",162565,2016,126760,2016
4,05,93,2016,Homicides,Victime,0,"0,0000000",141107,2016,134647,2016


Le fichier est chargé en précisant que le code département doit être lu comme du texte.  
Cela permet notamment de conserver les zéros initiaux des départements tels que `01`, `02` ou `03`.

## 3. Structure du jeu de données

In [5]:
print(f"Nombre de lignes : {delits.shape[0]:,}".replace(",", " "))
print(f"Nombre de colonnes : {delits.shape[1]}")
print(f"Nombre de départements : {delits['Code_departement'].nunique()}")
print(f"Nombre de régions : {delits['Code_region'].nunique()}")
print(f"Nombre d'années : {delits['annee'].nunique()}")
print(f"Nombre d'indicateurs : {delits['indicateur'].nunique()}")
print(f"Période couverte : {delits['annee'].min()} à {delits['annee'].max()}")

Nombre de lignes : 16 362
Nombre de colonnes : 11
Nombre de départements : 101
Nombre de régions : 18
Nombre d'années : 9
Nombre d'indicateurs : 18
Période couverte : 2016 à 2024


In [6]:
delits.columns.tolist()

['Code_departement',
 'Code_region',
 'annee',
 'indicateur',
 'unite_de_compte',
 'nombre',
 'taux_pour_mille',
 'insee_pop',
 'insee_pop_millesime',
 'insee_log',
 'insee_log_millesime']

Le jeu de données contient une ligne pour chaque combinaison :

**département × année × indicateur**.

Les 16 362 lignes correspondent bien à :

`101 départements × 9 années × 18 indicateurs`.

In [7]:
delits["indicateur"].drop_duplicates().sort_values().reset_index(drop=True)

0                           Cambriolages de logement
1           Destructions et dégradations volontaires
2     Escroqueries et fraudes aux moyens de paiement
3                                          Homicides
4                              Tentatives d'homicide
5                              Trafic de stupéfiants
6                               Usage de stupéfiants
7                         Usage de stupéfiants (AFD)
8                    Usage de stupéfiants (hors AFD)
9            Violences physiques hors cadre familial
10               Violences physiques intrafamiliales
11                               Violences sexuelles
12                                   Vols avec armes
13                  Vols d'accessoires sur véhicules
14                           Vols dans les véhicules
15                                  Vols de véhicule
16           Vols sans violence contre des personnes
17                           Vols violents sans arme
Name: indicateur, dtype: str

## 4. Contrôle de la qualité des données

### 4.1. Valeurs manquantes

In [8]:
delits.isnull().sum()

Code_departement       0
Code_region            0
annee                  0
indicateur             0
unite_de_compte        0
nombre                 0
taux_pour_mille        0
insee_pop              0
insee_pop_millesime    0
insee_log              0
insee_log_millesime    0
dtype: int64

Aucune valeur manquante n'est observée dans le fichier.

### 4.2. Doublons

In [9]:
nb_doublons = delits.duplicated(
    subset=["annee", "Code_departement", "indicateur"]
).sum()

print(f"Nombre de doublons sur la clé année-département-indicateur : {nb_doublons}")

Nombre de doublons sur la clé année-département-indicateur : 0


Une ligne devant correspondre à un indicateur, un département et une année, cette combinaison constitue la clé du jeu de données.  
Aucun doublon n'est observé sur cette clé.

## 5. Vérification et correction des types

In [10]:
delits.dtypes

Code_departement       string
Code_region             int64
annee                   int64
indicateur                str
unite_de_compte           str
nombre                  int64
taux_pour_mille           str
insee_pop               int64
insee_pop_millesime     int64
insee_log               int64
insee_log_millesime     int64
dtype: object

La colonne `taux_pour_mille` est reconnue comme du texte, car les décimales utilisent une virgule.  
Elle doit être convertie en variable numérique.

In [11]:
delits["taux_pour_mille"] = delits["taux_pour_mille"].str.replace(
    ",", ".", regex=False
)
delits["taux_pour_mille"] = delits["taux_pour_mille"].astype(float)

delits.dtypes

Code_departement        string
Code_region              int64
annee                    int64
indicateur                 str
unite_de_compte            str
nombre                   int64
taux_pour_mille        float64
insee_pop                int64
insee_pop_millesime      int64
insee_log                int64
insee_log_millesime      int64
dtype: object

In [12]:
delits[["nombre", "taux_pour_mille", "insee_pop", "insee_log"]].describe().round(2)

,nombre,taux_pour_mille,insee_pop,insee_log
count,16362.00,16362.00,16362.00,16362.00
mean,1794.08,2.43,667511.03,363306.83
std,4238.29,2.88,511525.34,260378.81
min,0.00,0.00,76422.00,60395.00
25%,157.00,0.43,283372.00,165115.00
50%,690.00,1.64,529374.00,295109.00
75%,1861.75,3.34,849583.00,481682.00
max,162378.00,74.99,2616909.00,1396753.00


La conversion a bien été effectuée. Les variables quantitatives peuvent maintenant être utilisées pour les analyses et les futurs KPI.

## 6. Sélection de la France métropolitaine

Le projet porte sur la France métropolitaine.  
Les régions d'outre-mer, identifiées par les codes `1`, `2`, `3`, `4` et `6`, sont donc retirées du périmètre.

In [13]:
codes_regions_outre_mer = [1, 2, 3, 4, 6]

delits_metropole = delits.loc[
    ~delits["Code_region"].isin(codes_regions_outre_mer)
].copy()

print(f"Nombre de lignes conservées : {len(delits_metropole):,}".replace(",", " "))
print(f"Nombre de départements : {delits_metropole['Code_departement'].nunique()}")
print(f"Nombre de régions : {delits_metropole['Code_region'].nunique()}")

Nombre de lignes conservées : 15 552
Nombre de départements : 96
Nombre de régions : 13


Après filtrage, le jeu de données contient **96 départements métropolitains** répartis dans **13 régions**.

## 7. Unités de compte et interprétation des taux

In [14]:
(
    delits_metropole[["indicateur", "unite_de_compte"]]
    .drop_duplicates()
    .sort_values(["unite_de_compte", "indicateur"])
    .reset_index(drop=True)
)

,indicateur,unite_de_compte
0,Cambriolages de logement,Infraction
1,Destructions et dégradations volontaires,Infraction
2,Vols avec armes,Infraction
3,Vols violents sans arme,Infraction
4,Trafic de stupéfiants,Mis en cause
5,Usage de stupéfiants,Mis en cause
6,Usage de stupéfiants (AFD),Mis en cause
7,Usage de stupéfiants (hors AFD),Mis en cause
8,Escroqueries et fraudes aux moyens de paiement,Victime
9,Homicides,Victime


Les indicateurs ne comptent pas tous exactement la même réalité : selon les cas, l'unité peut être une victime, une infraction, un véhicule ou un logement.

Un autre point de vigilance concerne le dénominateur des taux :

- la plupart des taux sont rapportés à la population ;
- le taux de cambriolages de logement est rapporté au nombre de logements.

Il ne serait donc pas rigoureux d'additionner directement les 18 taux pour construire un « taux global de délinquance ».  
Dans SQL et Power BI, les comparaisons seront réalisées indicateur par indicateur, avec possibilité de filtrer la catégorie d'infraction.

## 8. Synthèse de la préparation

À l'issue de cette première étape :

- le fichier contient 16 362 observations et 11 variables ;
- aucune valeur manquante n'a été détectée ;
- aucun doublon n'a été détecté sur la clé métier ;
- la variable `taux_pour_mille` a été convertie en nombre décimal ;
- le périmètre a été limité à la France métropolitaine ;
- les différences d'unités et de dénominateurs ont été identifiées.

Le jeu de données peut maintenant être exporté pour alimenter la base SQL et le dashboard Power BI.

In [15]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

delits_metropole.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Fichier exporté : {OUTPUT_PATH}")
print(f"Dimensions du fichier exporté : {delits_metropole.shape}")

Fichier exporté : ../data/processed/delinquance_metropole.csv
Dimensions du fichier exporté : (15552, 11)
